# 22 — One GeoAI result, GeoLibre and RustGIS

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/22_rustgis_geoai_bridge.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Widget/anywidget/LexCube branches need their source extensions; the Atlas shell supplies only Pyodide.

**Provenance:** `book/notebooks/22_rustgis_geoai_bridge.ipynb` at `777fa8965068b1e26b982162f17254ee2df7ec84`; original SHA-256 `3d22814b2947e96538bcd2e5cad526e4676dfcbeef6f30dca37289027d4d2460`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 22 — One GeoAI result, GeoLibre and RustGIS

**Question:** can we fit a simple geographic pattern in the browser, then inspect the same result in two mapping tools?

[Read What is RustGIS?](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/platforms/rustgis.html) for the illustrated introduction and credits. Run with **Shift+Enter**, from top to bottom. The data are **20 invented points**, not real facilities or patient locations. No account, API key, Ollama, native Rust or RustGIS installation is required.

![Python fits groups in JupyterLite; GeoJSON carries the result to separate GeoLibre and RustGIS browser apps.](assets/rustgis/bridge.svg)

JupyterLite runs Python through Pyodide. RustGIS Web runs a separate Rust/WebAssembly GIS. The native Python `rustgis` package needs ordinary CPython; **do not install it with micropip here**. GeoJSON is our portable data format, not a shared project or automatic synchronization protocol.


## Start here: the question behind the code

**Can we inspect the same geographic groups in GeoLibre and RustGIS?**

**What you will learn:** inspect → fit → export → compare. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** DBSCAN learns groups from invented locations. Exporting GeoJSON and changing map projection do not train a model.

**Why this is useful:** Portable data and a method card let learners compare tools without losing the meaning or limits of an analysis.

**Where the work happens:** Pyodide fits groups; separate GeoLibre and RustGIS web apps render maps. The native rustgis wheel is not a Pyodide package.

![One GeoAI result, two mapping tools. Read 20 invented locations and their longitude/latitude coordinates. Group nearby locations, then test a different distance. Save GeoJSON and a provenance card before opening another app. Inspect the published example in RustGIS; import your export for changed results.](assets/learning/22/infographic.svg)

### Watch the idea unfold

[[Source image continuation: Preview: One GeoAI result, two mapping tools — 32-second captioned walkthrough](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/assets/learning/22/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/22/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/22/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=22_rustgis_geoai_bridge.ipynb) · [Read this Book chapter](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/notebooks/22_rustgis_geoai_bridge.html) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/22/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **Inspect (0–8 s):** Read 20 invented locations and their longitude/latitude coordinates. Code landmark: `sample_sites()`.

2. **Fit (8–16 s):** Group nearby locations, then test a different distance. Code landmark: `fit_groups(sites, radius_m)`.

3. **Export (16–24 s):** Save GeoJSON and a provenance card before opening another app. Code landmark: `export_groups(grouped)`.

4. **Compare (24–32 s):** Inspect the published example in RustGIS; import your export for changed results. Code landmark: `public_view_url()`.


### Code guide 1 — Prepare the browser workbench

**Read the Python:** `sys.platform` selects Pyodide package loading. The helper is bundled with this notebook. RustGIS Web will run separately; no native rustgis wheel is installed here.

**Look for:** Scientific tools ready; no network data or AI account required after package loading.

**Pause and try:** Which software runs Python, and which software will display the exported map?


In [ ]:
import sys
if sys.platform == "emscripten":
    from pyodide_js import loadPackage
    await loadPackage(["numpy", "matplotlib", "scikit-learn", "ipython"])
import json
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
from rustgis_bridge import (sample_sites, validate_sites, fit_groups, group_counts,
                            plot_sites, export_groups, public_view_url)
print("Tools ready. The data will be invented locally; no model has run yet.")


## 1. Start with a map, not an answer

These points form two small grids and two isolated locations near London. They are deliberately invented so we can understand an algorithm without exposing anyone's location. A dot's position is a **measurement input in the example**, not evidence that a clinic exists there. The tile-free plot is a coordinate view, not a road or travel-time map.


### Code guide 2 — Inspect invented points

**Read the Python:** A FeatureCollection contains features. Each Point stores [longitude, latitude], an identifier and properties. validate_sites checks this lesson's coordinate schema. The scatter plot only displays data.

**Look for:** 20 invented locations; no fitted groups yet.

**Pause and try:** Find the coordinates and synthetic label on the first feature.


In [ ]:
sites = sample_sites()
validate_sites(sites)
print(f"{len(sites['features'])} invented points")
print(json.dumps(sites['features'][0], indent=2))
plot_sites(sites, "Invented points: mapping before machine learning")
plt.show()


## 2. Add one GeoAI step

**DBSCAN** is an unsupervised learning method: it computes groups from the locations without being given correct labels. The helper calls scikit-learn's `DBSCAN(eps=radius_m / EARTH_RADIUS_M, min_samples=3, metric="haversine", algorithm="brute").fit_predict(...)`. It first reverses GeoJSON longitude/latitude into latitude/longitude and converts degrees to radians.

The 350-meter setting is a chosen neighborhood distance, **not** a learned disease-transmission distance. Connected neighborhoods can make a group wider than 350 meters. A label of `-1` means unassigned at this setting; it does not mean an error or a low-risk place. No disease, population, roads, capacity or service-use data enter this model.


### Code guide 3 — Fit geographic groups

**Read the Python:** fit_groups runs scikit-learn DBSCAN. It reverses coordinates into [latitude, longitude], converts to radians and uses haversine distance. Three nearby points, including the point itself, define a core neighborhood.

**Look for:** At 350 meters: two groups of nine points and two unassigned points (-1).

**Pause and try:** Does a group identify disease risk? No: no disease observations or population denominators were supplied.


In [ ]:
radius_m = 350
grouped = fit_groups(sites, radius_m=radius_m)
print("Group label -> number of points:", group_counts(grouped))
plot_sites(grouped, f"DBSCAN location groups: {radius_m} m neighbors")
plt.show()


## 3. Ask how much the assumption matters

Try a smaller neighborhood. This cell prints comparison fits while keeping `grouped` unchanged for export. To export a new setting, change `radius_m` above and rerun that cell and the export cell. Group colors and numbers are identifiers, not severity ranks.

**Health-program transfer:** an analyst might explore candidate geographic groupings of public service locations. That still requires verified facilities, demand, roads, travel time, staffing and community review before making a service-placement decision. These invented groups establish none of those facts.


### Code guide 4 — Test the distance assumption

**Read the Python:** The loop fits fresh groups at two distances. It does not overwrite grouped, the result used by later export cells. Group numbers are arbitrary and are not stable identities across model settings.

**Look for:** 150 meters leaves all 20 points unassigned; 350 meters finds two groups.

**Pause and try:** Change radius_m in the previous cell and rerun from there to export a different result.


In [ ]:
for trial_radius in (150, 350):
    trial = fit_groups(sites, radius_m=trial_radius)
    print(f"{trial_radius} m:", group_counts(trial))


## 4. Export your result, including its limits

Run the cell, then **download both files**. In JupyterLite, use the links or right-click each file in the `exports` folder and select Download. Files inside the browser's notebook filesystem are not automatically visible to another tab.

Open [RustGIS Web](https://web.rustgis.app/?blank), use **Add Data** (or drag the downloaded `.geojson` onto the map), and inspect a feature's `id`, `group`, `neighbor_radius_m` and `synthetic` fields. Style categories by `group` if desired. Keep the provenance JSON alongside the data; it is a method card, not a layer. RustGIS may save its own `.rgis` project; do not treat that as a GeoLibre project.


### Code guide 5 — Save data with provenance

**Read the Python:** export_groups writes GeoJSON and a separate method card. FileLink creates download links; it does not publish the files or send them to another app. The JupyterLite file browser also offers Download.

**Look for:** Two files in exports; use Add Data or drag the GeoJSON into RustGIS Web.

**Pause and try:** Download both files. Can another person recover the coordinate order, model settings and synthetic-data label?


In [ ]:
data_path, provenance_path = export_groups(grouped)
print("Notebook files saved; download them before importing into another app.")
if sys.platform == "emscripten":
    display(FileLink(str(data_path)), FileLink(str(provenance_path)))
else:
    # Static Book outputs must not link to files created in a temporary test kernel.
    print(data_path, provenance_path)


## 5. Open a ready-made 2D or globe example

These links load the **published 350-meter snapshot** directly from GitHub Pages, which must be reachable and allow cross-origin requests. They do **not** upload or show your edits. To inspect changed results, use the file transfer above. A globe changes the view; there are no measured heights or 3D risk estimates in this dataset. If a hosted app or URL fails, use the saved plots and downloaded GeoJSON.


### Code guide 6 — Open the published example

**Read the Python:** public_view_url percent-encodes the hosted data URL. The globe link adds a projection setting. Both links open the fixed published 350-meter example, not your notebook's current export.

**Look for:** 2D and globe links to RustGIS Web; no model is refitted by changing projection.

**Pause and try:** Compare an id and group in the two views. Are there any measured heights?


In [ ]:
display(Markdown(
    f"[Open the published example in RustGIS (2D)]({public_view_url()}) · "
    f"[Open the same example on a globe]({public_view_url(globe=True)})"
))


## 6. Optional: inspect the same current result in GeoLibre

Change the flag only when you want the hosted interactive viewer. This uses the repository's browser adapter, with the same GeoJSON already computed in `grouped`. It does not refit a model. Re-run after changing the earlier result. Compare feature values across apps; styling and camera settings belong to each app.


### Code guide 7 — Compare an optional GeoLibre view

**Read the Python:** The optional branch sends the same GeoJSON into LiteMap. It runs only when you change the flag. A GeoLibre project and a RustGIS .rgis project are different formats; GeoJSON is the shared data.

**Look for:** Default run stays with saved plots and transfer links. Optional viewing needs packages and the hosted GeoLibre app.

**Pause and try:** Compare feature properties across apps rather than expecting identical styles or cameras.


In [ ]:
SHOW_GEOLIBRE = False
comparison_map = None
if SHOW_GEOLIBRE:
    if sys.platform == "emscripten":
        await loadPackage("micropip")
        import micropip
        await micropip.install(["geolibre==3.0.0", "anywidget"])
    from geolibre_lite import LiteMap
    comparison_map = LiteMap(center=(-0.11, 51.505), zoom=11)
    comparison_map.add_geojson(grouped, name="Invented DBSCAN groups")
    display(comparison_map)
else:
    print("Optional GeoLibre view is off. Plots, GeoJSON and RustGIS links remain available.")


## What did we learn?

| Operation | What changed? | Model fitting? |
|---|---|---|
| Plot the input points | A view of the supplied coordinates | No |
| Fit DBSCAN | Group membership derived from geographic neighbors | Yes, unsupervised learning |
| Change the radius and refit | The grouping assumption and possibly the labels | Yes |
| Open 2D, globe or GeoLibre | How an existing result is displayed | No |
| Export GeoJSON | Where data can be read | No |

For a separate **native CPython 3.12+** environment, the [RustGIS Python documentation](https://github.com/opengeos/RustGIS/blob/8b322b290448e468c8972df9b1f601d3a72f7d53/docs/python.md) describes `pip install rustgis`, `rustgis.Session()`, `session.add_geojson(...)` and `session.save("example.rgis")`. That route is not executed in this browser lesson.

## Sources and attribution

- **RustGIS:** Wu, Q., Rincón Albarracín, J. G., Filho, E., Hofbauer, D., Fitzgerald, M., & Strebel, D. (2026). *RustGIS: A fast, free and open-source GIS in pure Rust for mapping, analyzing and sharing geospatial data.* [Zenodo, doi:10.5281/zenodo.23633281](https://doi.org/10.5281/zenodo.23633281). Apache-2.0; [source and contributors](https://github.com/opengeos/RustGIS). Web launch parameters: [installation documentation, reviewed revision](https://github.com/opengeos/RustGIS/blob/8b322b290448e468c8972df9b1f601d3a72f7d53/docs/installation.md).
- **Companion teaching context:** James L. Tobias, [JupyterLite-RustGIS](https://github.com/jltobias/JupyterLite-RustGIS), reviewed 2026-10-30. Its notebooks and maps are linked, not copied into this lesson.
- **DBSCAN implementation:** [scikit-learn DBSCAN reference](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html). Geographic distance is interpreted in meters after conversion to angular distance.
- **GeoLibre:** Qiusheng Wu and contributors; [software, citation and data credits](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/platforms/geolibre.html).

The synthetic dataset, runtime illustration and lesson are authored for this repository. No external source dataset, health record, RustGIS screenshot or companion artwork is reproduced. Replacing a basemap or viewer does not change data attribution.


## Make sense of the result

![Does a globe make these groups more intelligent? No. DBSCAN fitted the groups before either map opened. Keep the data and method; choose a useful view.](assets/learning/22/cartoon.svg)

**Cartoon transcript:** Does a globe make these groups more intelligent? → No. DBSCAN fitted the groups before either map opened. → Keep the data and method; choose a useful view.

**Explain it to a colleague:** Will the launch link show a radius you just changed in the notebook?

<details><summary>Check your reasoning</summary>

No. The link opens the published 350-meter snapshot. Download your changed GeoJSON and import it into RustGIS Web.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.


## GeoLibre creator and software attribution

GeoLibre is developed by **Qiusheng Wu and contributors**. Recommended citation: Wu, Q. (2026). *GeoLibre: A lightweight, cloud-native GIS platform for visualizing, exploring, and analyzing geospatial data.* Zenodo. [doi:10.5281/zenodo.20785400](https://doi.org/10.5281/zenodo.20785400). See [the official citation guidance](https://geolibre.app/citation/) and [source and MIT license](https://github.com/opengeos/GeoLibre). These browser lessons pin the Python builders to `geolibre==3.0.0`; the independently hosted viewer may change. This repository supplies the JupyterLite adapter and teaching materials, not the upstream GIS. Keep the original dataset, demo-author and basemap credits when sharing a result.
